# ORF v4.0.0 — One-Cell Runner

Provide a **.zip of the project** (Google **Drive** path, a local `ZIP_PATH`, or the **upload** prompt) and the single code cell below will unzip it, locate `main.py`, install `requirements.txt`, optionally **activate real PQC encryption**, and execute the pipeline — streaming logs inline and listing the report artifacts (JSON / PDF / payload dossier).

## How to use
1. Edit the config at the **top of the cell** (mainly `MODE` and `SETUP_PQC`).
2. Run the cell.
3. Provide the project `.zip` (Drive path / `ZIP_PATH` / upload prompt).
4. Watch the live output; PDF/dossier downloads appear at the end (Colab).

## Two modes — set `MODE`
- **`"benchmark"` (default, OFFLINE):** runs the technique composer over the seed corpus in `data/`. Needs **no LLM endpoint**, so the cell always produces output.
- **`"run"` (LIVE):** audits a real SSE target. Payloads come from a **local Ollama attacker**, and every **VULNERABLE / COMPLIANT verdict is decided by a local LLM judge** (no API tokens).

## Judge — local, no token cost
Verdicts come purely from the judge model — configure it in the cell:
- `JUDGE_MODEL` — a strong instruction-tuned local model judges best (e.g. `qwen2.5`); empty = reuse the attacker model.
- `JUDGE_SAMPLES` / `JUDGE_TEMP` — self-consistency voting. **Strong judge → `1` / `0.0`** (accurate + fast); weak judge → `3` / `0.3`.
- `JUDGE_FEWSHOT` — few-shot exemplars in the judge prompt (on by default).

## Real PQC report encryption (optional) — `SETUP_PQC`
The report vault can encrypt with **ML-KEM-768 (Kyber) + AES-GCM**, but that crypto is **dormant** unless native `liboqs` is built *and* a keypair exists at the path the pipeline checks. Set `SETUP_PQC = True` (**Colab / Linux only**) and the cell will build native liboqs, generate that keypair, and run a verified encrypt→decrypt self-test — printing `✅ PQC ACTIVE`. A live `run` then writes an encrypted `SECURE_REPORT_*.pqc.json` instead of plaintext. On Windows/non-Linux it is a safe no-op (reports stay plaintext).

## Suite — set `SUITE`
`quick` (default — fast subset of both surfaces) · `llm` (OWASP LLM Top 10) · `agentic` (OWASP ASI T1–T15) · `all`.

## Embedding-lab mode — `MODE = "vvs"`
Runs the `vvs_test.py` harness against the embedding-attack-lab boundary service (`TESTING.md`) instead of `main.py`. It ignores the attacker/judge and PQC settings and never bootstraps a model — it drives the HTTP attacks against `VVS_BASE_URL` (default `https://your-target.example`). Choose the attack with `VVS_CMD` (`smoke` / `kplus` / `attack-a`–`attack-d` / `boundary` / `all`).


In [ ]:
# ============================================================================
# ORF v4.0.0 — ONE-CELL RUNNER :  (Drive/Upload) ZIP -> Unzip -> Install -> [PQC] -> Execute
# ----------------------------------------------------------------------------
# Run THIS single cell. It will:
#   1. Get a project .zip (Google Drive path, a local ZIP_PATH, or an upload prompt).
#   2. Extract into ./orf_workspace/ and auto-locate the folder with main.py.
#   3. pip install requirements.txt (best-effort).
#   4. (optional) Activate REAL PQC: build native liboqs + make the keypair.
#   5. Execute the pipeline, streaming logs live.
#   6. List the report artifacts (JSON / PDF / dossier / SECURE_REPORT_*.pqc.json).
# ============================================================================

import os, sys, zipfile, shutil, subprocess
from pathlib import Path

# ---- Config you can tweak --------------------------------------------------
WORKDIR      = Path("orf_workspace")   # where the archive is extracted
ZIP_PATH     = ""                      # local/Drive path to a .zip -> SKIPS the upload prompt
INSTALL_DEPS = True                    # pip install requirements.txt after extraction
MOUNT_DRIVE  = False                   # Colab: mount Google Drive (then set ZIP_PATH=/content/drive/...)
SETUP_PQC    = False                   # Colab/Linux ONLY: build native liboqs + keys so reports are PQC-ENCRYPTED

# MODE picks what to execute:
#   "benchmark" = fully OFFLINE self-test over the seed corpus in data/.
#                 Needs NO LLM endpoint; always produces output. (default)
#   "run"       = LIVE audit against the target below, judged by a LOCAL model.
#   "vvs"       = run vvs_test.py against the embedding-attack-lab (TESTING.md).
MODE = "benchmark"

# --- VVS EMBEDDING-LAB TARGET (only used when MODE="vvs") --------------
# Runs the vvs_test.py harness against the embedding-attack-lab boundary
# service from TESTING.md (default: the live your-target.example deployment). This
# mode is self-contained HTTP: it ignores every Ollama / attacker / judge and
# SSE-target setting in this cell and never bootstraps a local model.
VVS_BASE_URL = "https://your-target.example"   # embedding-lab base URL
VVS_CMD      = "all"    # smoke | kplus | attack-a | attack-b | attack-c | attack-d | boundary | all
VVS_ARMS     = "A,B,C,D"                  # storage arms to exercise
VVS_DIM      = 2560                       # M1 = Qwen3-Embedding-4B (embedding dim)
VVS_DEBUG    = False                      # log each HTTP request to stderr

# ---- LIVE-audit settings (used only when MODE == "run") --------------------
SUITE          = "quick"                           # quick | llm | agentic | all
TARGET_URL     = "https://your-chat-host.example"  # SSE chat endpoint under test
TARGET_COOKIE  = ""                                # optional Cookie header for the target
TARGET_KEY     = ""                                # optional Bearer token for the target

# Attacker = LOCAL Ollama that GENERATES payloads:
ATTACKER_BASE  = "http://localhost:11434"
ATTACKER_MODEL = "dolphin-llama3"

# Judge = LOCAL model that DECIDES vulnerable/compliant (no token cost).
#   * A strong instruction-tuned model judges best (e.g. "qwen2.5").
#   * Strong judge -> JUDGE_SAMPLES=1, JUDGE_TEMP=0.0 (accurate + fast).
#   * Weak judge   -> keep voting (JUDGE_SAMPLES=3, JUDGE_TEMP=0.3).
JUDGE_BASE     = "http://localhost:11434"
JUDGE_MODEL    = ""                                # "" = reuse the attacker model
JUDGE_SAMPLES  = 3                                 # self-consistency votes (1 = single pass)
JUDGE_TEMP     = 0.3                               # per-sample temperature (>0 if SAMPLES>1)
JUDGE_FEWSHOT  = True                              # few-shot exemplars in the judge prompt


def _build_run_args():
    """Assemble the main.py argv from the config above."""
    if MODE != "run":
        return ["benchmark", "--use-seeds"]
    a = [
        "run",
        "--suite", SUITE,
        "--target-provider", "sse", "--base-url", TARGET_URL,
        "--attacker-provider", "ollama", "--attacker-base-url", ATTACKER_BASE,
        "--model", ATTACKER_MODEL,
        "--judge-provider", "ollama", "--judge-base-url", JUDGE_BASE,
        "--judge-samples", str(JUDGE_SAMPLES),
        "--judge-temperature", str(JUDGE_TEMP),
    ]
    if JUDGE_MODEL:
        a += ["--judge-model", JUDGE_MODEL]
    if not JUDGE_FEWSHOT:
        a += ["--judge-no-fewshot"]
    if TARGET_COOKIE:
        a += ["--target-cookie", TARGET_COOKIE]
    if TARGET_KEY:
        a += ["--target-api-key", TARGET_KEY]
    return a

RUN_ARGS = _build_run_args()

# ---- 0. (optional) Mount Google Drive --------------------------------------
if MOUNT_DRIVE:
    try:
        from google.colab import drive            # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped:", exc)

# ---- 1. Obtain the zip -----------------------------------------------------
zip_path = Path(ZIP_PATH) if ZIP_PATH else None
if zip_path is None:
    try:
        from google.colab import files            # type: ignore
        print("Select the project .zip to upload ...")
        uploaded = files.upload()
        zips = [n for n in uploaded if n.lower().endswith(".zip")]
        if not zips:
            raise SystemExit("No .zip file was uploaded.")
        zip_path = Path(zips[0])
    except ModuleNotFoundError:
        # Not on Colab: try an ipywidgets uploader, else ask for ZIP_PATH.
        try:
            import ipywidgets as w
            from IPython.display import display
            up = w.FileUpload(accept=".zip", multiple=False)
            display(up)
            if not up.value:
                raise SystemExit("Pick a .zip in the widget above, then re-run this cell.")
            item = (list(up.value.values())[0] if isinstance(up.value, dict) else up.value[0])
            name = (item.get("metadata", {}) or {}).get("name") or item.get("name", "upload.zip")
            zip_path = Path(name)
            zip_path.write_bytes(item["content"])
        except ImportError:
            raise SystemExit("No Colab/ipywidgets here - set ZIP_PATH to your local .zip and re-run.")

zip_path = zip_path.resolve()
print(f"Using archive: {zip_path}")
if not zipfile.is_zipfile(zip_path):
    raise SystemExit(f"Not a valid zip file: {zip_path}")

# ---- 2. Extract ------------------------------------------------------------
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORKDIR)
print(f"Extracted to: {WORKDIR.resolve()}")

# ---- 3. Locate the project root (shallowest main.py) -----------------------
mains = sorted(WORKDIR.rglob("main.py"), key=lambda p: len(p.parts))
if not mains:
    raise SystemExit("Could not find main.py inside the archive.")
project_root = mains[0].parent
print(f"Project root: {project_root}")

# ---- 4. Install dependencies (best-effort) ---------------------------------
req = project_root / "requirements.txt"
if INSTALL_DEPS and req.exists():
    print(f"Installing dependencies from {req} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)
else:
    print("Skipping dependency install (no requirements.txt or disabled).")

# ---- 4b. (optional) Activate REAL PQC: native liboqs + keypair -------------
# The report vault uses ML-KEM-768 (Kyber) + AES-GCM, but the crypto is DORMANT
# unless (a) native liboqs is built and (b) a keypair exists where the pipeline
# looks (/workspace/audit_authority.pub). This block does both, then proves it
# with an encrypt->decrypt self-test. Linux/Colab only (native build needs a C
# toolchain); safe no-op elsewhere.
if SETUP_PQC and MODE != "vvs":
    import platform, json as _json
    if platform.system() != "Linux":
        print("SETUP_PQC skipped: native liboqs needs a Linux build toolchain (use Colab). Reports stay plaintext.")
    else:
        print("Activating PQC: installing cmake/gcc/git + liboqs and building the native lib (~1-2 min)...")
        subprocess.run("apt-get -qq install -y cmake gcc git".split(), check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycryptodome", "liboqs-python"], check=False)
        os.makedirs("/workspace", exist_ok=True)
        # Keypair at the EXACT path run_standalone_pipeline checks with os.path.exists().
        subprocess.run(
            [sys.executable, "main.py", "keygen",
             "--pub", "/workspace/audit_authority.pub",
             "--key", "/workspace/audit_authority.key"],
            cwd=str(project_root), check=False,
        )
        if not os.path.exists("/workspace/audit_authority.pub"):
            print("  WARNING: keygen produced no public key (liboqs build likely failed) -> reports will be PLAINTEXT.")
        else:
            # End-to-end proof via the armor/unlock subcommands.
            _tin, _tenc, _tout = "/workspace/_pqc_in.json", "/workspace/_pqc.pqc.json", "/workspace/_pqc_out.json"
            with open(_tin, "w") as _f:
                _json.dump([{"pqc_selftest": "ok", "n": 123}], _f)
            subprocess.run(
                [sys.executable, "main.py", "armor", "--input", _tin,
                 "--pub", "/workspace/audit_authority.pub", "--output", _tenc],
                cwd=str(project_root), check=False,
            )
            subprocess.run(
                [sys.executable, "main.py", "unlock", "--input", _tenc,
                 "--key", "/workspace/audit_authority.key", "--output", _tout],
                cwd=str(project_root), check=False,
            )
            _encrypted = False
            if os.path.exists(_tenc):
                with open(_tenc) as _f:
                    _encrypted = "pqc_kem_envelope" in _json.load(_f)
            _restored = False
            if os.path.exists(_tout):
                with open(_tout) as _f:
                    _restored = _json.load(_f) == [{"pqc_selftest": "ok", "n": 123}]
            if _encrypted and _restored:
                print("  \u2705 PQC ACTIVE - Kyber768+AES round-trip verified. A 'run' will emit SECURE_REPORT_*.pqc.json.")
            else:
                print("  WARNING: PQC NOT active (fell back to plaintext). Check the liboqs build logs above.")

# ---- 5. Execute, streaming stdout+stderr live ------------------------------
if MODE == "vvs":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "requests"], check=False)
    cmd = [sys.executable, "vvs_test.py",
           "--base-url", VVS_BASE_URL, "--arms", VVS_ARMS, "--dim", str(VVS_DIM)]
    if VVS_DEBUG:
        cmd += ["--debug"]
    cmd += [VVS_CMD]
else:
    run_args = RUN_ARGS.split() if isinstance(RUN_ARGS, str) else list(RUN_ARGS)
    cmd = [sys.executable, "main.py"] + run_args
print("\nExecuting:", " ".join(cmd))
print("=" * 74)
proc = subprocess.Popen(
    cmd, cwd=str(project_root),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("=" * 74)
print(f"Process exited with code {proc.returncode}")

# ---- 6. Surface the generated report artifacts -----------------------------
# PQC-encrypted reports land in /workspace/secure_vault/; PDF/dossier + plaintext
# fallback land in <project>/secure_vault/. Scan both.
vault_dirs = [project_root / "secure_vault", Path("/workspace/secure_vault")]
seen_any = False
for vault in vault_dirs:
    if not vault.exists():
        continue
    files = [f for f in sorted(vault.glob("*")) if f.is_file()]
    if not files:
        continue
    seen_any = True
    print(f"\nReport artifacts in {vault}:")
    for f in files:
        tag = "  [PQC-ENCRYPTED]" if f.name.endswith(".pqc.json") else ""
        print(f"  - {f.name}  ({f.stat().st_size} bytes){tag}")
    # In Colab, offer the PDF + dossier for download.
    try:
        from google.colab import files as _dl      # type: ignore
        for f in files:
            if f.suffix.lower() in (".pdf", ".md"):
                _dl.download(str(f))
    except Exception:
        pass
if not seen_any:
    print("\nNo report artifacts were produced.")
